In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")

torch.manual_seed(42)

# 1) Patch Embedding

**The core idea of ViT** (Dosovitskiy et al., 2020)

Images are split into non-overlapping patches, each flattened and projected
to an embedding dimension. This converts an image into a *sequence of tokens*
that a standard Transformer encoder can process.

**Why this matters**
- No spatial inductive bias: attention attends to *all* patches equally,
  regardless of distance. CNNs can only attend within the kernel window.
- Scales well with data: more data → attention learns better long-range
  dependencies; CNNs plateau faster.
- Weakness: requires more data than CNNs to learn spatial patterns from scratch
  (addressed by DeiT in Section 6).

**Sample I/O**
```
Input image    : (batch_num, n_channels=1, H=28, W=28)
Patch size p=4 : grid of 7×7 = 49 patches, each of size 4×4×1 = 16 features
After linear   : (batch_num, num_patches=49, embed_dim)
After CLS token: (batch_num, num_patches+1=50, embed_dim)
```

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Splits an image into patches and linearly embeds each patch.

    This is equivalent to a Conv2d with kernel_size=patch_size and
    stride=patch_size — a single non-overlapping convolution.

    Args:
        img_size    : Input image height/width (assumed square)
        patch_size  : Size of each square patch
        in_channels : Number of input channels
        embed_dim   : Embedding dimension for each patch token
    """

    def __init__(
        self,
        img_size: int = 28,
        patch_size: int = 4,
        in_channels: int = 1,
        embed_dim: int = 128,
    ) -> None:
        super().__init__()

        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"

        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size

        # 1 non-overlapping conv == split into patches + linear projection
        # (batch_num, in_channels, H, W) → (batch_num, embed_dim, H//p, W//p)
        self.proj = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (batch_num, in_channels, H, W)

        Returns:
            patches : (batch_num, num_patches, embed_dim)
        """
        # (batch_num, in_channels, H, W) → (batch_num, embed_dim, H//p, W//p)
        x = self.proj(x)
        # (batch_num, embed_dim, H//p, W//p) → (batch_num, embed_dim, num_patches)
        x = x.flatten(2)
        # (batch_num, embed_dim, num_patches) → (batch_num, num_patches, embed_dim)
        return x.transpose(1, 2)


# ── Verify ─────────────────────────────────────────────────────────────────────
pe = PatchEmbedding(img_size=28, patch_size=4, in_channels=1, embed_dim=128)
x_img = torch.zeros(4, 1, 28, 28)
with torch.no_grad():
    patches = pe(x_img)
print(f"Image    : {x_img.shape}")  # (4, 1, 28, 28)
print(f"Patches  : {patches.shape}")  # (4, 49, 128)  ← 49 = 7×7 patches

# 2) Multi-Head Self-Attention

Patch tokens go through the same scaled dot-product attention as text tokens, with two
differences: no causal mask (every patch attends to every patch) and a sequence length fixed by
the image grid ($(H/p)(W/p)$ patches plus CLS). The module below uses PyTorch's fused
`scaled_dot_product_attention` kernel.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Bidirectional multi-head self-attention over patch tokens."""

    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.dropout = dropout
        # (batch_num, num_tokens, embed_dim) -> (batch_num, num_tokens, 3 * embed_dim)
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, num_tokens, embed_dim = x.shape
        # (batch_num, num_tokens, 3 * embed_dim) -> 3 x (batch_num, num_heads, num_tokens, head_dim)
        q, k, v = (
            self.qkv(x)
            .view(batch_num, num_tokens, 3, self.num_heads, self.head_dim)
            .permute(2, 0, 3, 1, 4)
        )
        # no mask: attention over all patches
        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0
        )
        # (batch_num, num_heads, num_tokens, head_dim) -> (batch_num, num_tokens, embed_dim)
        return self.out_proj(out.transpose(1, 2).reshape(batch_num, num_tokens, embed_dim))


mhsa = MultiHeadSelfAttention(embed_dim=128, num_heads=4)
x_attn = torch.zeros(2, 50, 128)
with torch.no_grad():
    out_attn = mhsa(x_attn)
print(f"MHSA: {tuple(x_attn.shape)} -> {tuple(out_attn.shape)}")

# 3) ViT Encoder Block and Full ViT

**Transformer encoder block**
```
LayerNorm → MHSA → Residual (+x)
LayerNorm → MLP  → Residual (+x)
```

**Pre-norm vs post-norm**
The original Transformer (Vaswani et al.) uses post-norm (LN after residual).
ViT uses *pre-norm* (LN before attention). Pre-norm is more stable for deep
networks and is now the standard in vision transformers.

**Classification token (CLS)**
A learnable CLS token is prepended to the patch sequence.
After all encoder blocks, only the CLS token is passed to the classification head.
The CLS token aggregates information from all patches via attention.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    One ViT encoder block: pre-norm MHSA + pre-norm MLP, both with residual.
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        # MLP expands to mlp_ratio × embed_dim then contracts back
        hidden_dim = int(embed_dim * mlp_ratio)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # GELU preferred over ReLU in transformers
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm MHSA with residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = x + self.attn(self.norm1(x))
        # Pre-norm MLP with residual
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        x = x + self.mlp(self.norm2(x))
        return x


class ViT(nn.Module):
    """
    Vision Transformer for image classification.

    Architecture:
        Patch embedding → [CLS] token → Positional embedding
        → num_layers × TransformerEncoderBlock → LayerNorm
        → CLS token → classification head
    """

    def __init__(
        self,
        img_size: int = 28,
        patch_size: int = 4,
        in_channels: int = 1,
        num_classes: int = 10,
        embed_dim: int = 128,
        num_layers: int = 6,
        num_heads: int = 4,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        # Learnable CLS token — one vector prepended to the patch sequence
        # (1, 1, embed_dim) — broadcast over batch
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embeddings — one per patch + one for CLS
        # (1, num_patches+1, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        self.pos_drop = nn.Dropout(dropout)

        # Stack of transformer encoder blocks
        self.blocks = nn.ModuleList(
            [
                TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
                for _ in range(num_layers)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

        # Classification head: only uses the CLS token output
        # (batch_num, embed_dim) → (batch_num, num_classes)
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialise weights — important for transformer stability
        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num = x.size(0)

        # Convert image to patch token sequence
        # (batch_num, in_channels, H, W) → (batch_num, num_patches, embed_dim)
        x = self.patch_embed(x)

        # Expand and prepend CLS token
        # (1, 1, embed_dim) → (batch_num, 1, embed_dim)
        cls = self.cls_token.expand(batch_num, -1, -1)
        # (batch_num, num_patches+1, embed_dim)
        x = torch.cat([cls, x], dim=1)

        # Add learned positional embeddings
        # (batch_num, num_patches+1, embed_dim)
        x = self.pos_drop(x + self.pos_embed)

        # Apply transformer encoder blocks
        for block in self.blocks:
            # (batch_num, num_patches+1, embed_dim) → same
            x = block(x)

        x = self.norm(x)

        # Extract CLS token and classify
        # (batch_num, embed_dim)
        cls_out = x[:, 0]
        # (batch_num, num_classes)
        return self.head(cls_out)


# ── Shape dry-run ──────────────────────────────────────────────────────────────
vit = ViT(
    img_size=28,
    patch_size=4,
    in_channels=1,
    num_classes=10,
    embed_dim=128,
    num_layers=6,
    num_heads=4,
)
x_vit = torch.zeros(4, 1, 28, 28)
with torch.no_grad():
    out_vit = vit(x_vit)

print(f"ViT input  : {x_vit.shape}")  # (4, 1, 28, 28)
print(f"ViT output : {out_vit.shape}")  # (4, 10)
print(f"Parameters : {sum(p.numel() for p in vit.parameters()):,}")

# 4) Train ViT on MNIST

**Important observation for research**
ViT needs significantly more data than CNNs to learn spatial inductive biases
from scratch. On MNIST (60 000 images, 28×28) it reaches good accuracy, but
would be clearly outperformed by a CNN of similar parameter count on a harder
dataset without ImageNet pretraining.

## 4.1 Training

In [ ]:
# Data loading for MNIST
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)
mnist_tr = torchvision.datasets.MNIST(
    "./data", True, download=True, transform=transform
)
mnist_vl = torchvision.datasets.MNIST(
    "./data", False, download=True, transform=transform
)
tr_loader = DataLoader(mnist_tr, batch_size=128, shuffle=True, num_workers=2)
vl_loader = DataLoader(mnist_vl, batch_size=256, shuffle=False, num_workers=2)


def train_vit(model, tr_loader, vl_loader, n_epochs=5, lr=3e-4):
    """Standard training loop with AdamW + cosine schedule."""
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0
        for X, y in tr_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 1, 28, 28) → (batch_num, 10)
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X.size(0)
        scheduler.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for X, y in vl_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                correct += (model(X).argmax(1) == y).sum().item()

        tl = running_loss / len(tr_loader.dataset)
        acc = correct / len(vl_loader.dataset)
        print(f"Epoch {epoch}/{n_epochs} | loss={tl:.4f} | val_acc={acc:.3f}")


torch.manual_seed(0)
vit_model = ViT(
    img_size=28,
    patch_size=4,
    in_channels=1,
    num_classes=10,
    embed_dim=128,
    num_layers=6,
    num_heads=4,
    dropout=0.1,
)
train_vit(vit_model, tr_loader, vl_loader, n_epochs=5, lr=3e-4)

## 4.2 Dynamic Resolution — Interpolating Positional Embeddings

**The problem with fixed positional embeddings**

Our `ViT` stores a `pos_embed` tensor of shape `(1, num_patches + 1, embed_dim)`,
where `num_patches = (img_size // patch_size) ** 2`.
If the input image has a different size at inference time, the number of patches
changes — and the stored positional embeddings no longer have the right shape.

**Why this matters for research**
- Pretrained on 224×224 but want to evaluate on 384×384 (higher res, better accuracy)
- Multi-scale inference for dense prediction tasks (segmentation, detection)
- Fine-tuning on datasets with different aspect ratios

**The fix: bicubic interpolation of positional embeddings**

The patch positional embeddings encode a 2D spatial grid.
We can treat them as a low-resolution 2D feature map and **bicubically upsample**
them to any target resolution — the same operation used to resize images.

```
pos_embed (excl. CLS) :  (1, old_grid_h, old_grid_w, embed_dim)
                              ↓  F.interpolate bicubic
                         (1, new_grid_h, new_grid_w, embed_dim)
```

This is how the original ViT paper, DINOv2, and all Hugging Face ViT models
handle variable-resolution inference. The HF `ViTImageProcessor` does it
automatically via `interpolate_pos_encodings=True`.

In [ ]:
def interpolate_pos_embed(
    pos_embed: torch.Tensor,
    old_grid_size: int,
    new_grid_size: int,
    embed_dim: int,
) -> torch.Tensor:
    """
    Bicubically interpolates patch positional embeddings to a new grid resolution.

    The CLS token embedding is left unchanged; only the patch tokens are resized.
    Uses bicubic interpolation (same as image resizing) because positional
    embeddings are smooth 2D functions of spatial position.

    Args:
        pos_embed     : (1, old_num_patches + 1, embed_dim)  — full stored embedding
        old_grid_size : sqrt(old_num_patches), e.g. 7 for a 28×28 image with patch=4
        new_grid_size : Target grid size, e.g. 14 for a 56×56 image with patch=4
        embed_dim     : Embedding dimension

    Returns:
        pos_embed_interp : (1, new_num_patches + 1, embed_dim)  — interpolated embedding
    """
    if old_grid_size == new_grid_size:
        return pos_embed  # no interpolation needed

    # Separate CLS token (first token) from patch tokens
    # (1, 1, embed_dim)
    cls_token_embed = pos_embed[:, :1, :]
    # (1, old_num_patches, embed_dim)
    patch_embed = pos_embed[:, 1:, :]

    # Reshape patch tokens to a 2D spatial grid for interpolation
    # (1, old_num_patches, embed_dim) → (1, old_grid_size, old_grid_size, embed_dim)
    patch_embed = patch_embed.view(1, old_grid_size, old_grid_size, embed_dim)

    # Rearrange to (1, embed_dim, old_grid_size, old_grid_size) for F.interpolate
    patch_embed = patch_embed.permute(0, 3, 1, 2)

    # Bicubic upsample to new spatial resolution
    # (1, embed_dim, old_grid_size, old_grid_size) → (1, embed_dim, new_grid_size, new_grid_size)
    patch_embed = F.interpolate(
        patch_embed.float(),
        size=(new_grid_size, new_grid_size),
        mode="bicubic",
        align_corners=False,
    ).to(pos_embed.dtype)

    # Rearrange back to (1, new_num_patches, embed_dim)
    # (1, embed_dim, new_grid_size, new_grid_size) → (1, new_grid_size^2, embed_dim)
    patch_embed = patch_embed.permute(0, 2, 3, 1)
    patch_embed = patch_embed.view(1, new_grid_size * new_grid_size, embed_dim)

    # Concatenate CLS token back at position 0
    # (1, new_num_patches + 1, embed_dim)
    return torch.cat([cls_token_embed, patch_embed], dim=1)


class ViTDynamic(ViT):
    """
    ViT with dynamic resolution support via positional embedding interpolation.

    Inherits the full ViT architecture; overrides only the forward method to
    detect resolution changes and interpolate pos_embed accordingly.

    This is identical to how ViT-Base/16 handles resolution changes in the
    official JAX implementation and all Hugging Face ViT checkpoints.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, _, H, W = x.shape

        # Compute number of patches in this forward pass
        new_grid_h = H // self.patch_embed.patch_size
        new_grid_w = W // self.patch_embed.patch_size
        new_num_patches = new_grid_h * new_grid_w

        # Training resolution grid size (stored at init)
        old_num_patches = self.patch_embed.num_patches
        old_grid_size = int(old_num_patches**0.5)

        # Convert image to patch tokens
        # (batch_num, in_channels, H, W) → (batch_num, new_num_patches, embed_dim)
        x = self.patch_embed(x)

        # Prepend CLS token
        # (batch_num, new_num_patches + 1, embed_dim)
        cls = self.cls_token.expand(batch_num, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # Interpolate positional embeddings if resolution has changed
        # (1, old_num_patches+1, embed_dim) → (1, new_num_patches+1, embed_dim)
        pos_embed = interpolate_pos_embed(
            self.pos_embed,
            old_grid_size=old_grid_size,
            new_grid_size=new_grid_h,  # assumes square grid
            embed_dim=self.pos_embed.shape[-1],
        )

        # Add (possibly interpolated) positional embeddings
        # (batch_num, new_num_patches+1, embed_dim)
        x = self.pos_drop(x + pos_embed)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # Use CLS token for classification
        # (batch_num, num_classes)
        return self.head(x[:, 0])


# ── Demonstration: same model weights, three different input resolutions ───────
torch.manual_seed(0)
vit_dyn = ViTDynamic(
    img_size=28,
    patch_size=4,
    in_channels=1,
    num_classes=10,
    embed_dim=128,
    num_layers=6,
    num_heads=4,
)
# Load the weights we trained in Section 4 (same architecture)
vit_dyn.load_state_dict(vit_model.state_dict())
vit_dyn.eval()

print("Resolution → grid size → pos_embed shape → output shape")
print("-" * 60)
for res in [28, 56, 112]:
    x_test = torch.zeros(2, 1, res, res)
    grid = res // 4
    with torch.no_grad():
        out = vit_dyn(x_test)
    expected_pos_dim = grid * grid + 1
    print(
        f"  {res}×{res}  →  {grid}×{grid}={grid**2} patches  →  "
        f"pos_embed interpolated to ({1}, {expected_pos_dim}, 128)  →  {out.shape}"
    )

## 4.3 Non-Square and Rectangular Inputs

Standard ViT assumes square images and a square patch grid.
For rectangular inputs (common in medical imaging, satellite imagery, documents),
we need to interpolate with `new_grid_h ≠ new_grid_w`.

The `interpolate_pos_embed` function above already handles this because
`F.interpolate` with `size=(h, w)` supports non-square targets.
The only adjustment needed is tracking `grid_h` and `grid_w` separately.

In [ ]:
class ViTDynamicRect(ViTDynamic):
    """
    ViT with full non-square resolution support.
    Tracks grid_h and grid_w separately to handle rectangular inputs.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_num, _, H, W = x.shape
        p = self.patch_embed.patch_size
        new_grid_h = H // p
        new_grid_w = W // p
        new_num_patches = new_grid_h * new_grid_w
        old_grid_size = int(self.patch_embed.num_patches**0.5)

        # (batch_num, in_channels, H, W) → (batch_num, new_num_patches, embed_dim)
        x = self.patch_embed(x)
        cls = self.cls_token.expand(batch_num, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # Rectangular interpolation: old square grid → new (h, w) grid
        cls_emb = self.pos_embed[:, :1, :]
        patch_emb = self.pos_embed[:, 1:, :]
        embed_dim = patch_emb.shape[-1]

        # (1, old_num_patches, embed_dim) → (1, embed_dim, old_g, old_g)
        patch_emb = patch_emb.view(1, old_grid_size, old_grid_size, embed_dim)
        patch_emb = patch_emb.permute(0, 3, 1, 2)

        # (1, embed_dim, old_g, old_g) → (1, embed_dim, new_grid_h, new_grid_w)
        patch_emb = F.interpolate(
            patch_emb.float(),
            size=(new_grid_h, new_grid_w),
            mode="bicubic",
            align_corners=False,
        ).to(self.pos_embed.dtype)

        # (1, embed_dim, new_grid_h, new_grid_w) → (1, new_num_patches, embed_dim)
        patch_emb = patch_emb.permute(0, 2, 3, 1).view(1, new_num_patches, embed_dim)
        pos_embed = torch.cat([cls_emb, patch_emb], dim=1)

        x = self.pos_drop(x + pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


# ── Non-square demo ────────────────────────────────────────────────────────────
vit_rect = ViTDynamicRect(
    img_size=28,
    patch_size=4,
    in_channels=1,
    num_classes=10,
    embed_dim=128,
    num_layers=6,
    num_heads=4,
)
vit_rect.load_state_dict(vit_model.state_dict())
vit_rect.eval()

print("Non-square inputs:")
for H, W in [(28, 28), (28, 56), (56, 112), (112, 56)]:
    x_rect = torch.zeros(2, 1, H, W)
    grid_h, grid_w = H // 4, W // 4
    with torch.no_grad():
        out_rect = vit_rect(x_rect)
    print(
        f"  Input {H}×{W}  →  grid {grid_h}×{grid_w}={grid_h*grid_w} patches  →  out {out_rect.shape}"
    )


# ── Visualise positional embedding interpolation ───────────────────────────────
# Show how each patch position's embedding changes across resolutions
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
resolutions = [28, 56, 112]
for ax, res in zip(axes, resolutions):
    grid = res // 4
    old_g = 7  # trained on 28×28 with patch=4

    cls_e = vit_dyn.pos_embed[:, :1, :]
    patch_e = vit_dyn.pos_embed[:, 1:, :]
    dim = patch_e.shape[-1]

    patch_e = patch_e.view(1, old_g, old_g, dim).permute(0, 3, 1, 2)
    patch_e = F.interpolate(
        patch_e.float(), size=(grid, grid), mode="bicubic", align_corners=False
    )
    patch_e = patch_e.permute(0, 2, 3, 1).view(grid, grid, dim)

    # Show norm of each patch position embedding as a proxy for its strength
    # (grid, grid)
    norms = patch_e.norm(dim=-1).detach().numpy()
    im = ax.imshow(norms, cmap="viridis")
    ax.set_title(f"Res {res}×{res}  →  {grid}×{grid} grid")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Positional embedding norms after bicubic interpolation")
plt.tight_layout()
plt.show()

# 5) Transfer Learning from a Pretrained ViT

A ViT trained from scratch needs far more data than a CNN; a ViT **pretrained** on ImageNet-21K
transfers well to small datasets. The target here is **Beans** (leaf disease photos, 3 classes,
1,034 training images), far from MNIST and close enough to natural images for ImageNet features
to help. ViT-Base/16 has 86M parameters, 12 layers, 768 hidden dim, and 197 tokens per
224×224 image (196 patches + CLS).

Three strategies, from cheapest to most expensive:

| Strategy | What is trained | When it is the right call |
|---|---|---|
| Linear probe | A classical classifier on frozen, cached CLS features | Very little data; a quick diagnostic of how good the features already are |
| Head-only training | The new classification head, backbone frozen | Little data, no time for full fine-tuning |
| Full fine-tuning | Everything, with a head warm-up and smaller learning rates for earlier layers | A few thousand images or a domain shift from the pretraining data |

In [ ]:
import time

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from transformers import ViTForImageClassification, ViTImageProcessor

MODEL_NAME = "google/vit-base-patch16-224-in21k"
beans_ds = load_dataset("AI-Lab-Makerere/beans")
labels = beans_ds["train"].features["labels"].names
print(f"Classes: {labels} | train {len(beans_ds['train'])}, val {len(beans_ds['validation'])}")

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)


def new_vit() -> ViTForImageClassification:
    """Pretrained ViT-B/16 with a freshly initialised 3-class head."""
    return ViTForImageClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=dict(enumerate(labels)),
        label2id={c: i for i, c in enumerate(labels)},
    ).to(DEVICE)


def batches(split: str, batch_num: int = 32, shuffle: bool = False):
    """(pixel_values, labels) batches of a Beans split, processed for ViT."""
    ds = beans_ds[split].shuffle(seed=0) if shuffle else beans_ds[split]
    for start in range(0, len(ds), batch_num):
        rows = ds[start : start + batch_num]
        # list of PIL images -> (batch_num, 3, 224, 224)
        pixels = processor(images=rows["image"], return_tensors="pt")["pixel_values"]
        yield pixels.to(DEVICE), torch.tensor(rows["labels"], device=DEVICE)


vit_pretrained = new_vit()
total = sum(p.numel() for p in vit_pretrained.parameters())
head = sum(p.numel() for p in vit_pretrained.classifier.parameters())
print(f"Parameters: {total:,} total, {head:,} in the new head ({100 * head / total:.3f}%)")

## 5.1 Linear probe on frozen features

Run every image through the frozen backbone **once**, cache the CLS vector, and fit a logistic
regression. No backpropagation through the network, and the result says how linearly separable
the classes already are in the pretrained feature space: if this is already high, fine-tuning has
little headroom.

In [ ]:
@torch.no_grad()
def cls_features(model: ViTForImageClassification, split: str) -> tuple[np.ndarray, np.ndarray]:
    """Frozen CLS embeddings (num_images, 768) and labels for one split."""
    model.eval()
    feats, ys = [], []
    for pixels, y in batches(split):
        # (batch_num, 197, 768) -> CLS token (batch_num, 768)
        feats.append(model.vit(pixel_values=pixels).last_hidden_state[:, 0].float().cpu())
        ys.append(y.cpu())
    return torch.cat(feats).numpy(), torch.cat(ys).numpy()


start = time.time()
X_train, y_train = cls_features(vit_pretrained, "train")
X_val, y_val = cls_features(vit_pretrained, "validation")
probe = LogisticRegression(max_iter=2000).fit(X_train, y_train)
results = {"linear probe": probe.score(X_val, y_val)}
print(f"Features {X_train.shape} in {time.time() - start:.0f}s")
print(f"Linear probe val accuracy: {results['linear probe']:.1%}")

## 5.2 Head-only training, then full fine-tuning

**Phase 1** trains only the new head with the backbone frozen. A random head sends large, noisy
gradients into the backbone if everything is trainable from the first step; warming the head up
first avoids damaging pretrained features.

**Phase 2** unfreezes everything with **layer-wise learning-rate decay**: layer $\ell$ of $L$
gets $\eta_\ell = \eta \cdot \gamma^{L-\ell}$ with $\gamma < 1$, so the early layers (generic
edges and textures) move least and the last layers and head adapt most.

In [ ]:
def evaluate(model: ViTForImageClassification) -> float:
    """Validation accuracy."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for pixels, y in batches("validation"):
            correct += (model(pixel_values=pixels).logits.argmax(-1) == y).sum().item()
            total += y.numel()
    return correct / total


def train_epochs(model: ViTForImageClassification, optimizer, epochs: int) -> None:
    """Plain cross-entropy training over the Beans training split."""
    for epoch in range(epochs):
        model.train()
        start = time.time()
        for pixels, y in batches("train", batch_num=16, shuffle=True):
            loss = F.cross_entropy(model(pixel_values=pixels).logits, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        print(f"  epoch {epoch + 1}: last loss {loss.item():.3f}, val acc {evaluate(model):.1%} ({time.time() - start:.0f}s)")


def layerwise_groups(model: ViTForImageClassification, lr: float, decay: float) -> list[dict]:
    """Optimizer parameter groups with lr * decay^(num_layers - layer) per encoder layer."""
    num_layers = model.config.num_hidden_layers
    groups = [
        {"params": model.classifier.parameters(), "lr": lr},
        {"params": model.vit.layernorm.parameters(), "lr": lr},
        {"params": model.vit.embeddings.parameters(), "lr": lr * decay ** (num_layers + 1)},
    ]
    for i, layer in enumerate(model.vit.layers):
        groups.append({"params": layer.parameters(), "lr": lr * decay ** (num_layers - i)})
    return groups


model = new_vit()
for p in model.vit.parameters():
    p.requires_grad = False
print("Phase 1: head only")
train_epochs(model, torch.optim.AdamW(model.classifier.parameters(), lr=1e-3), epochs=2)
results["head only"] = evaluate(model)

for p in model.vit.parameters():
    p.requires_grad = True
print("Phase 2: all layers, layer-wise lr decay 0.75")
train_epochs(model, torch.optim.AdamW(layerwise_groups(model, lr=5e-5, decay=0.75), weight_decay=0.05), epochs=2)
results["head warm-up + full fine-tune"] = evaluate(model)

## 5.3 Comparison

| If the target data... | Use |
|---|---|
| is tiny (hundreds of images) or you need an answer in minutes | Linear probe on cached features |
| is small and close to the pretraining domain | Head-only training |
| has thousands of images or differs from the pretraining domain | Head warm-up, then full fine-tuning with layer-wise lr decay |

A strong linear probe means the pretrained features already separate the classes; the gap between
the probe and full fine-tuning is what adapting the backbone buys on this dataset. On Beans the
frozen ImageNet-21K features are already enough (the probe matches full fine-tuning), and the
head-only run trails the probe only because two epochs of SGD on a fresh head have not reached the
optimum that logistic regression solves for directly. Fine-tuning pays off when the target domain
is further from the pretraining data than leaf photos are from ImageNet.

In [ ]:
for name, acc in results.items():
    print(f"{name:<32} val accuracy {acc:.1%}")

# 6) DeiT and Swin Transformer

## 6.1 DeiT — Data-Efficient Image Transformers

**The problem DeiT solves**
Original ViT requires 300M+ images (JFT-300M) to outperform CNNs.
DeiT (Touvron et al., 2021) achieves competitive results using only ImageNet
(1.2M images) through:

1. **Aggressive augmentation**: CutMix, MixUp, RandAugment
2. **Knowledge distillation**: a teacher CNN (e.g., RegNetY-16GF) supervises
   the ViT student via a special *distillation token*

**Distillation token**
DeiT adds a second learnable token alongside CLS:
- CLS token: trained with standard cross-entropy against true label
- Distillation token: trained with hard/soft label from teacher model
- At inference: average the two token predictions

## 6.2 Swin Transformer — Hierarchical ViT

**Why hierarchical matters**
Original ViT uses fixed-size patches with global attention — quadratic in number
of tokens, and no multi-scale features (needed for detection/segmentation).

Swin (Liu et al., 2021) introduces:
1. **Window attention**: attention within local 7×7 windows → linear complexity
2. **Shifted windows**: windows shift each layer to enable cross-window connections
3. **Hierarchical stages**: like ResNet, spatial resolution decreases and channels
   increase through the network — enabling FPN-style multi-scale feature maps

Swin-T is the standard backbone for many detection/segmentation papers.

In [ ]:
from transformers import AutoImageProcessor, SwinForImageClassification

# ── Load pretrained Swin-Tiny ─────────────────────────────────────────────────
swin_processor = AutoImageProcessor.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224"
)
swin_model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224"
).to(DEVICE)

print("Swin-Tiny configuration:")
print(f"  Parameters   : {sum(p.numel() for p in swin_model.parameters()):,}")
print(f"  Patch size   : {swin_model.config.patch_size}")
print(f"  Window size  : {swin_model.config.window_size}")
print(f"  Depths       : {swin_model.config.depths}  (blocks per stage)")
print(f"  Num heads    : {swin_model.config.num_heads}")
print(f"  ImageNet-1K classes: {swin_model.config.num_labels}")

# Sample inference
sample_img = beans_ds["train"][5]["image"]
inputs_swin = swin_processor(images=sample_img, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs_swin = swin_model(**inputs_swin)
    # logits: (1, 1000)  — ImageNet-1K classes
    top5_idx = outputs_swin.logits[0].topk(5).indices.tolist()
    imagenet_labels = swin_model.config.id2label

print("\nSwin-Tiny top-5 ImageNet predictions on a Beans image:")
for idx in top5_idx:
    print(f"  {imagenet_labels[idx]}")

# ── Model comparison table ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Model':<20} {'Params':>10} {'Attn type':>18} {'Scales':>8}")
print("=" * 60)
models_info = [
    ("ViT-B/16", "86M", "Global (full)", "Single"),
    ("DeiT-B/16", "86M", "Global + distill", "Single"),
    ("Swin-T", "28M", "Local window+shift", "Multi"),
    ("Swin-B", "88M", "Local window+shift", "Multi"),
]
for name, params, attn, scales in models_info:
    print(f"{name:<20} {params:>10} {attn:>18} {scales:>8}")